# Практика №3.

## Задание

1. Провести статистическую обработку текста.
2. Определить энтропию, приходящуюся в среднем на одну букву, длину кода при равномерном кодировании и избыточность.
3. Построить схему алфавитного кодирования для однобуквенных сочетаний методом Хаффмана. Найти среднюю длину элементарного кода, эффективность сжатия. Закодировать текст. Декодировать текст.
4. Построить схему алфавитного кодирования для двухбуквенных сочетаний методом Шеннона-Фано. Найти среднюю длину элементарного кода, эффективность сжатия, сравнить с результатами для однобуквенных сочетаний. Закодировать текст. Декодировать текст.

In [24]:
import math
import pickle
import string
from collections import Counter

## Обработка входного текста

In [25]:
with open('some_text.txt', 'r', encoding='utf-8') as file:
    text = file.read()

In [26]:
text = text.lower()  # Приведение к нижнему регистру
text = text.translate(str.maketrans('', '', string.punctuation + '\n'))  # Удаление знаков препинания
text = text.replace(" ", "")  # Удаление пробелов

## Основные функции

In [27]:
# Подсчет частот символов
def calculate_frequencies(text: str) -> Counter[str]:
    single_char_freq = Counter(text)  # Частоты однобуквенных сочетаний
    bigrams = [text[i:i+2] for i in range(len(text)-1)]  # Двухбуквенные сочетания
    bigram_freq = Counter(bigrams)
    return single_char_freq, bigram_freq

In [28]:
# Вычисление энтропии текста
def calculate_entropy(freq_dict: dict, total_count: int) -> float:
    entropy = 0
    for freq in freq_dict.values():
        p = freq / total_count
        entropy -= p * math.log2(p)
    return entropy

In [29]:
# Алгоритм Шеннона-Фано
class HuffmanFanoNode:
    def __init__(self, probability: float, data, left=None, right=None) -> None:
        self.probability = probability
        self.data = data
        self.left = left
        self.right = right


def convert_to_huffman_nodes(freq_dict: dict, total_count: int) -> list[HuffmanFanoNode]:
    nodes = []
    for char, count in freq_dict.items():
        probability = count / total_count
        node = HuffmanFanoNode(probability, char)
        nodes.append(node)
    nodes.sort(key=lambda node: node.probability, reverse=True)
    return nodes


def build_huffman_fano_tree(node_list: list[HuffmanFanoNode]) -> HuffmanFanoNode:
    while len(node_list) > 1:
        first_node = node_list.pop()
        second_node = node_list.pop()
        new_node = HuffmanFanoNode(first_node.probability + second_node.probability,
                        None, first_node, second_node)
        node_list.append(new_node)
        node_list.sort(key=lambda node: node.probability, reverse=True)

    return node_list[0]


def generate_huffman_codes(node, prefix="", codebook={}):
    if node.left is None and node.right is None:
        codebook[node.data] = prefix
    else:
        generate_huffman_codes(node.left, prefix + "0", codebook)
        generate_huffman_codes(node.right, prefix + "1", codebook)

In [30]:
# Кодирование текста
def encode(message: str, codebook: dict[str, str]) -> str:
    encoded_message = ""
    for letter in message:
        encoded_message += codebook[letter]

    return encoded_message

# Декодирование текста
def decode(encoded_message: str, codebook: dict[str, str]) -> str:
    decoding_codebook = dict((code, char) for char, code in codebook.items())
    decoded_message = ""
    bits = ""
    for bit in encoded_message:
        bits += bit
        if decoding_codebook.get(bits) is not None:
            decoded_message += decoding_codebook[bits]
            bits = ""

    return decoded_message

In [31]:
# Кодирование текста с сохранением в бинарный файл
def encode_to_file(message: str, codebook: dict[str, str], filename: str) -> None:
    # Создание бинарного представления закодированного сообщения
    encoded_bits = ''.join(codebook[letter] for letter in message)
    
    # Преобразование битов в байты
    byte_array = bytearray()
    for i in range(0, len(encoded_bits), 8):
        byte = int(encoded_bits[i:i + 8], 2)  # Преобразуем 8 бит в целое число
        byte_array.append(byte)
    
    # Сохранение закодированного сообщения и кодовой книги в бинарный файл
    with open(filename, 'wb') as f:
        pickle.dump((byte_array, codebook), f)

# Декодирование текста из бинарного файла
def decode_from_file(filename: str) -> str:
    # Загрузка закодированного сообщения и кодовой книги из бинарного файла
    with open(filename, 'rb') as f:
        byte_array, codebook = pickle.load(f)

    # Преобразование байтов в битовую строку
    encoded_bits = ''.join(f'{byte:08b}' for byte in byte_array)
    
    decoding_codebook = {code: char for char, code in codebook.items()}
    decoded_message = ""
    bits = ""
    for bit in encoded_bits:
        bits += bit
        if bits in decoding_codebook:
            decoded_message += decoding_codebook[bits]
            bits = ""

    return decoded_message

### 1. Провести статистическую обработку текста.

In [32]:
single_char_freq, bigram_freq = calculate_frequencies(text)

In [33]:
# Генерация вероятностной схемы
print(" char | n     | w     ")
print("------+-------+-------")
for char, count in single_char_freq.items():
    print(f"{char:>5} | {count:>5} | {(count / len(text)):>6.4f}")

 char | n     | w     
------+-------+-------
    l |  4366 | 0.0597
    o |  3081 | 0.0422
    r |  4129 | 0.0565
    e |  8315 | 0.1138
    m |  3567 | 0.0488
    i |  7535 | 0.1031
    p |  1812 | 0.0248
    s |  6166 | 0.0844
    u |  6720 | 0.0919
    d |  2174 | 0.0297
    t |  6129 | 0.0839
    a |  6048 | 0.0827
    c |  3106 | 0.0425
    n |  4502 | 0.0616
    g |   906 | 0.0124
    b |   908 | 0.0124
    f |   749 | 0.0102
    v |  1239 | 0.0170
    q |   993 | 0.0136
    h |   416 | 0.0057
    j |    68 | 0.0009
    x |   159 | 0.0022


### 2. Определить энтропию, приходящуюся в среднем на одну букву, длину кода при равномерном кодировании и избыточность.

In [34]:
# Энтропия
single_char_entropy = calculate_entropy(single_char_freq, len(text))
print(f"Энтропия: {single_char_entropy:.4f} бит на символ")

# Длина кода при равномерном кодировании
uniform_code_length = math.log2(len(single_char_freq))
uniform_code_length_ceil = math.ceil(uniform_code_length)
print(f"Длина кода при равномерном кодировании: {uniform_code_length_ceil} бит")

# Избыточность
redundancy = uniform_code_length - single_char_entropy
print(f"Избыточность: {redundancy:.4f}")

redundancy_2 = 1 - uniform_code_length / uniform_code_length_ceil
print(f"Избыточность округления: {redundancy_2:.4f}")

redundancy_3 = 1 - single_char_entropy / math.log2(len(single_char_freq))
print(f"Информационная избыточность: {redundancy_3:.4f}")

Энтропия: 3.9962 бит на символ
Длина кода при равномерном кодировании: 5 бит
Избыточность: 0.4633
Избыточность округления: 0.1081
Информационная избыточность: 0.1039


### 3. Построить схему алфавитного кодирования для однобуквенных сочетаний методом Хаффмана. Найти среднюю длину элементарного кода, эффективность сжатия. Закодировать текст. Декодировать текст.

In [35]:
# Генерация схемы кодирования по алгоритму Хаффмана
huffman_nodes = convert_to_huffman_nodes(single_char_freq, len(text))
huffman_tree = build_huffman_fano_tree(huffman_nodes)
codebook = {}
generate_huffman_codes(huffman_tree, codebook=codebook)

In [36]:
# Схема кодирования по алгоритму Хаффмана
print(" char | code")
print("------+-----------")
for char, code in codebook.items():
    print(f"{char:>5} | {code}")

 char | code
------+-----------
    u | 000
    i | 001
    m | 0100
    g | 010100
    b | 010101
    d | 01011
    e | 011
    r | 1000
    l | 1001
    n | 1010
    q | 101100
    v | 101101
    o | 10111
    a | 1100
    t | 1101
    s | 1110
    c | 11110
    j | 111110000
    x | 111110001
    h | 11111001
    f | 1111101
    p | 111111


In [37]:
# Средняя длина кодового слова
avg_code_length = sum([single_char_freq[char] / len(text) * len(codebook[char]) for char in single_char_freq])
print(f"Средняя длина кодового слова: {avg_code_length:.4f}")

# Коэффициент эффективности
efficiency_factor = single_char_entropy / avg_code_length
print(f"Коэффициент эффективности: {efficiency_factor:.4f}")

Средняя длина кодового слова: 4.0349
Коэффициент эффективности: 0.9904


In [38]:
# Кодирование текста
encoded_text = encode(text, codebook)
print("Закодированный текст:", encoded_text)

# Декодирование текста
decoded_text = decode(encoded_text, codebook)
print("Декодированный текст:", decoded_text)

Закодированный текст: ...


In [39]:
# Кодирование текста в бинарный файл
encode_to_file(text, codebook, "archived.bin")

# Декодирование текста из бинарного файла
decoded_text = decode_from_file("archived.bin")
print("Декодированный текст:", decoded_text)

Декодированный текст: ...


In [40]:
compression_factor = len(encoded_text) / (len(text) * 8) * 100
print(f"Коэффициент сжатия: {compression_factor:.4f}%")

Коэффициент сжатия: 50.4366%


### Построить схему алфавитного кодирования для двухбуквенных сочетаний методом Хаффмана. Найти среднюю длину элементарного кода, эффективность сжатия, сравнить с результатами для однобуквенных сочетаний. Закодировать текст. Декодировать текст.

In [41]:
# Генерация схемы кодирования по алгоритму Хаффмана
bigram_huffman_nodes = convert_to_huffman_nodes(bigram_freq, len(text)-1)
bigram_huffman_tree = build_huffman_fano_tree(bigram_huffman_nodes)
bigram_codebook = {}
generate_huffman_codes(bigram_huffman_tree, codebook=bigram_codebook)

In [42]:
# Схема кодирования по алгоритму Хаффмана
print(" chars | code")
print("-------+-----------")
for chars, code in bigram_codebook.items():
    print(f"{chars:>6} | {code}")

 chars | code
-------+-----------
    la | 000000
    uf | 000001000000
    uu | 000001000001
    dt | 00000100001
    ai | 0000010001
    tf | 000001001
    rn | 00000101
    bi | 00000110
    ro | 00000111
    um | 000010
    ul | 000011
    ss | 0001000
    as | 0001001
    nv | 000101000
    nn | 0001010010
    dl | 0001010011
    fi | 000101010
    sf | 000101011
    ru | 00010110
    gu | 00010111
    es | 000110
    am | 000111
    ae | 0010000
    lu | 0010001
    du | 0010010
    nm | 0010011000
    bl | 0010011001
    ph | 001001101
    ij | 00100111000000
    lg | 001001110000010
    uh | 001001110000011
    dh | 0010011100001
    ou | 001001110001
    mj | 0010011100100
    oo | 0010011100101
    xe | 0010011100110
    uq | 0010011100111
    vu | 0010011101
    ps | 001001111
    is | 00101
    ui | 0011000
    ap | 00110010
    mv | 001100110
    op | 00110011100
    mb | 001100111010
    lf | 001100111011
    rv | 0011001111
    al | 0011010
    co | 0011011
    tp | 0011

In [43]:
# Средняя длина кодового слова
avg_code_length = sum([bigram_freq[char] / (len(text)-1) * len(bigram_codebook[char]) for char in bigram_freq])
print(f"Средняя длина кодового слова: {avg_code_length:.4f}")

# Коэффициент эффективности
bigram_entropy = calculate_entropy(bigram_freq, len(text)-1)
bigram_efficiency_factor = bigram_entropy / avg_code_length
print(f"Коэффициент эффективности: {bigram_efficiency_factor:.4f}")

Средняя длина кодового слова: 7.4214
Коэффициент эффективности: 0.9957


In [44]:
# Кодирование текста
bigram_encoded_text = encode(text, codebook)
print("Закодированный текст:", bigram_encoded_text)

# Декодирование текста
bigram_decoded_text = decode(bigram_encoded_text, codebook)
print("Декодированный текст:", bigram_decoded_text)

Закодированный текст: ...


In [45]:
# Кодирование текста в бинарный файл
encode_to_file(text, codebook, "archived.bin")

# Декодирование текста из бинарного файла
bigram_decoded_text = decode_from_file("archived.bin")
print("Декодированный текст:", bigram_decoded_text)

Декодированный текст: ...


In [46]:
bigram_compression_factor = len(bigram_encoded_text) / (len(text) * 8) * 100
print(f"Коэффициент сжатия: {bigram_compression_factor:.4f}%")

Коэффициент сжатия: 50.4366%
